# 4. Inheritance

Inheritance in Python looks similar to Java, but with one massive difference: Python supports **multiple inheritance**. Java limits you to single class inheritance + interfaces; Python has no such restriction.

This notebook covers: basic inheritance, `super()`, method overriding, multiple inheritance, MRO (Method Resolution Order), type checking, `__init_subclass__`, and preventing inheritance.

### 4.1 Basic Inheritance & `super()`

**☕ JAVA:**
```java
public class Dog extends Animal {
    public Dog(String name, String breed) {
        super(name);       // Call parent constructor
        this.breed = breed;
    }
}
```

**🐍 PYTHON:** Use `class Dog(Animal):` instead of `extends`. Same `super()` concept, but no need to pass `self` to `super()` (Python 3 magic).

In [1]:
class Animal:
    def __init__(self, name: str):
        self.name = name
   
    def speak(self) -> str:
        return "Some sound"
   
    def __str__(self) -> str:
        return f"{self.__class__.__name__}({self.name})"
 
class Dog(Animal):
    def __init__(self, name: str, bread: str):
        super().__init__(name)
        self.bread = bread
 
dog = Dog("Buddy", "Labrador")
print(f"Name: {dog.name}")
print(f"Bread: {dog.bread}")
print(f"Speak: {dog.speak()}")
print(dog)

Name: Buddy
Bread: Labrador
Speak: Some sound
Dog(Buddy)


### 4.2 Method Overriding

**☕ JAVA:** `@Override` annotation is optional but recommended.

**🐍 PYTHON:** No `@Override` annotation exists. Just define a method with the same name — it automatically overrides. You can call `super().method()` to invoke the parent's version.

In [8]:
class Animal:
    def __init__(self, name: str):
        self.name = name
   
    def speak(self) -> str:
        return "..."
   
    def describe(self) -> str:
        return f"{self.name} says {self.speak()}"
   
class Dog(Animal):
    def speak(self): # Override - no annotation needed!
        return "Woof!"
 
class Cat(Animal):
    def speak(self):
        return "Meow"
 
class Parrot(Animal):
    def speak(self):
        return f"{super().speak()} Polly wants a carrot."
    
animals: list[Animal] = [Dog("Buddy"), Cat("Whiskers"), Parrot("Polly")]
for animal in animals:
    print(f"{animal.describe()}")

my_dog = Dog("Buddy")
my_cat = Cat("Whiskers")
my_parrot = Parrot("Polly")    

print(f"isinstance(my_dog, Dog): {isinstance(my_dog, Dog)}")  # True
print(f"isinstance(my_dog, Animal): {isinstance(my_dog, Animal)}")  # True
print(f"isinstance(my_dog, Cat): {isinstance(my_dog, Cat)}")  # False 

print(f"type(my_dog): {type(my_dog) == Dog}")
print(f"type(my_dog): {type(my_dog) == Animal}")  # <class '__main__.Dog'>

Buddy says Woof!
Whiskers says Meow
Polly says ... Polly wants a carrot.
isinstance(my_dog, Dog): True
isinstance(my_dog, Animal): True
isinstance(my_dog, Cat): False
type(my_dog): True
type(my_dog): False


### 4.3 Multiple Inheritance — Java Can't Do This!

**☕ JAVA:** Only single class inheritance. Interfaces provide partial workaround:
```java
public class FlyingFish extends Fish implements Flyable { ... }
```

**🐍 PYTHON:** Full multiple inheritance — a class can inherit from **multiple** parent classes:
```python
class FlyingFish(Fish, Flyer): ...
```

In [9]:
class Swimmer:
    def swim(self) -> str:
        return "Swimming"
 
class Flyer:
    def fly(self) -> str:
        return "Flying"
 
class Runner:
    def run(self) -> str:
        return "Running"
   
class Duck(Swimmer, Flyer, Runner):
    def __init__(self, name: str):
        self.name = name
 
duck = Duck("Donald")
print(f"{duck.name} can:")
print(f"  {duck.swim()}")
print(f"  {duck.fly()}")
print(f"  {duck.run()}")

Donald can:
  Swimming
  Flying
  Running


### 4.4 MRO — Method Resolution Order

**☕ JAVA:** No need — single inheritance means there's only one path to resolve.

**🐍 PYTHON:** With multiple inheritance, the same method might exist in multiple parents. Python uses **C3 Linearization** to determine a consistent order. View it with `ClassName.__mro__` or `ClassName.mro()`.

The **Diamond Problem** occurs when class D inherits from B and C, which both inherit from A:

In [15]:
class A:
  def greet(self) -> str:
    return "Hello from A"
  
class B(A):
  def greet(self) -> str:
    return "Hello from B"
  
class C(A):
  def greet(self) -> str:
    return "Hello from C"
  
class D(B, C):
  pass

d = D()
print(d.greet())  # Output: Hello from B

print(f"MRO: {[cls.__name__ for cls in D.__mro__]}")

#b = B()
#print(b.greet())
#c = C()
#print(c.greet())     

Hello from B
MRO: ['D', 'B', 'C', 'A', 'object']


> ⚠️ **Key insight:** `super()` doesn't always call the **direct parent** — it calls the **next class in the MRO**. This is why `B.greet()` calls `C.greet()` (not `A.greet()`) when called from `D`.

### 4.5 `isinstance()` vs `type()` — A Common Gotcha

**☕ JAVA:** `obj instanceof Dog` checks the full hierarchy.

**🐍 PYTHON:** `isinstance(obj, Dog)` and `issubclass(Dog, Animal)` both accept tuples for checking multiple types.

> ⚠️ **Gotcha:** `type(obj) == Class` checks the **exact** type only — it does NOT consider inheritance. Always prefer `isinstance()`.

### 4.6 `__init_subclass__` — Hook Into Subclassing

**☕ JAVA:** No equivalent — you can't execute code when a class is subclassed.

**🐍 PYTHON:** `__init_subclass__` is called automatically when a class is subclassed. Great for plugin registration, validation, or automatic setup.

### 4.7 Preventing Inheritance — `@final`

**☕ JAVA:** `final class String { ... }` — cannot be subclassed.

**🐍 PYTHON:** Python 3.8+ provides `typing.final` for two purposes:

| Use | Java | Python |
|-----|------|--------|
| Prevent subclassing | `final class Foo` | `@final class Foo` |
| Prevent overriding | `final void method()` | `@final def method()` |

> ⚠️ `@final` is a **type-checker hint only** (enforced by mypy/pyright). Python itself won't stop subclassing at runtime. For runtime enforcement, combine with `__init_subclass__`.

In [17]:
from typing import final

class Base:
    @final
    def critical_method(self) -> str:
        return "Don't touch me!"
    
    def extencible_method(self) -> str:
        return "Override me."
    
class Child(Base):
    # This will raise a TypeError at runtime
    # def critical_method(self) -> str:
    #     return "Trying to override"
    
    def critical_method(self) -> str:
        return "Do something else"

    def extencible_method(self) -> str:
        return "Everything is ok!"

c = Child()
print(f"Critical: {c.critical_method()}")
print(f"Extencible: {c.extencible_method()}")      

Critical: Do something else
Extencible: Everything is ok!


## Duck Typing

In [18]:
class Dog:
    def speak(self):
        return "Woof!"

class Cat:
    def speak(self):
        return "Meow!"

class Duck:
    def speak(self):
        return "Quack!"        

def animal_speak(animal):
    print(animal.speak())

dog = Dog()
cat = Cat()
duck = Duck()

animal_speak(dog)  # Output: Woof!
animal_speak(cat)  # Output: Meow!
animal_speak(duck)  # Output: Quack!


Woof!
Meow!
Quack!


---

## 🧪 Try It Yourself

**Exercise 1:** Create a `Vehicle` base class with `make` and `year`. Create `Car(Vehicle)` and `Truck(Vehicle)` subclasses that override a `describe()` method.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a diamond inheritance: `class ElectricAmphibious(ElectricVehicle, AmphibiousVehicle)` where both parents inherit from `Vehicle`. Print the MRO.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a `Handler` base class that auto-registers subclasses using `__init_subclass__`. Each subclass should specify a `handles` keyword (e.g., `class ImageHandler(Handler, handles='image')`). Add a class method `get_handler(media_type)` that returns the right handler.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Inherit | `class Dog extends Animal` | `class Dog(Animal):` |
| Call parent constructor | `super(name)` | `super().__init__(name)` |
| Override | `@Override` annotation | Just redefine the method |
| Call parent method | `super.method()` | `super().method()` |
| Multiple inheritance | ❌ Not supported | ✅ `class D(B, C):` |
| Interfaces | `implements Runnable` | Not needed — use ABC or Protocol |
| Diamond problem | Can't happen | Solved by MRO (C3 linearization) |
| `super()` target | Always direct parent | Next in MRO (may skip parent!) |
| Type check | `obj instanceof Dog` | `isinstance(obj, Dog)` |
| Exact type check | N/A | `type(obj) == Dog` (no inheritance!) |
| Multi-type check | Chained `instanceof` | `isinstance(obj, (Dog, Cat))` |
| Class check | `Dog.class.isAssignableFrom(...)` | `issubclass(Dog, Animal)` |
| Hook into subclassing | Not possible | `__init_subclass__(**kwargs)` |
| Prevent subclassing | `final class Foo` | `@final` (type-checker) or `__init_subclass__` |
| Prevent overriding | `final void method()` | `@final` (type-checker only) |